In [13]:
from dataclasses import dataclass
from pathlib import Path

# Fetch

In [2]:
import kagglehub

/home/cube/source/dhbw/ml-digits/model/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
temp_dir = Path.cwd() / ".temp"

if temp_dir.parent.name != "model":
    raise RuntimeError()

temp_dir.mkdir(exist_ok=True)
temp_dir

PosixPath('/home/cube/source/dhbw/ml-digits/model/.temp')

In [4]:
dataset_path = (
    Path(
        kagglehub.dataset_download(
            "metricasecuador/handwritten-digits-version-1-hwd-v1",
            output_dir=str(temp_dir / "dataset"),
        )
    )
    / "HWD-V1"
)
dataset_path

PosixPath('/home/cube/source/dhbw/ml-digits/model/.temp/dataset/HWD-V1')

# PyTorch

In [5]:
import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


(null): No such file or directory
(null): No such file or directory


# Dataset

In [7]:
RESOLUTION = 32

In [8]:
class DigitDataset(Dataset):
    def __init__(self, *, r: int):
        self.r = r
        self.items = [
            (item, int(item.parent.name))
            for item in list(dataset_path.glob("HWD-V1-Standard/*/*.png"))
        ]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx: int):
        item_path, label = self.items[idx]

        img = Image.open(item_path).convert("L")
        img = ImageOps.invert(img)

        img = img.resize((self.r, self.r))
        img.save(temp_dir / "test_img.png")

        data = np.array(img.get_flattened_data(), dtype=np.float32) / 255
        tensor = torch.tensor(data, dtype=torch.float32)
        tensor = tensor.view(1, self.r, self.r)

        return tensor, label


ds = DigitDataset(r=RESOLUTION)
ds.__getitem__(99999)

(tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]]),
 6)

# Model

In [19]:
class VerySimpleLinearModel(nn.Module):
    def __init__(self, *, r: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),  # (1, R, R) -> RxR
            nn.Linear(r * r, 10),  # RxR -> 10 digit scores
            nn.Softmax()
        )

    def forward(self, x):
        return self.net(x)

In [22]:
@dataclass(kw_only=True)
class TrainResult:
    dataset: Dataset
    model: nn.Module
    epoch: int
    loss: float
    accuracy: float


def train(
    *, dataset: DigitDataset, model: nn.Module, epochs: int, echo=True
) -> TrainResult:
    loader = DataLoader[DigitDataset](dataset, batch_size=32, shuffle=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total_loss = 0
        correct = 0

        for X_batch, Y_batch in loader:
            Y_pred: torch.Tensor = model(X_batch)
            loss = loss_fn(Y_pred, Y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * X_batch.size(0)
            correct += (Y_pred.argmax(1) == Y_batch).sum().item()

        n = len(dataset)
        calc_loss = total_loss / n
        calc_accuracy = correct / n

        if echo:
            print(
                f"Epoch {epoch + 1:2d}  loss={total_loss / n:.4f}  acc={correct / n:.2%}"
            )

    return TrainResult(
        dataset=dataset,
        model=model,
        epoch=epoch,
        loss=calc_loss,
        accuracy=calc_accuracy,
    )

In [ ]:
dataset = DigitDataset(r=RESOLUTION)
loader = DataLoader[DigitDataset](dataset, batch_size=32, shuffle=True)

result = train(dataset=dataset, model=VerySimpleLinearModel(r=RESOLUTION), epochs=30)

result


Epoch  1  loss=1.5262  acc=95.96%
Epoch  2  loss=1.4901  acc=97.68%
Epoch  3  loss=1.4857  acc=97.94%
Epoch  4  loss=1.4835  acc=98.10%
Epoch  5  loss=1.4820  acc=98.21%
Epoch  6  loss=1.4809  acc=98.30%
Epoch  7  loss=1.4801  acc=98.37%
Epoch  8  loss=1.4794  acc=98.40%
Epoch  9  loss=1.4789  acc=98.45%
Epoch 10  loss=1.4785  acc=98.49%
Epoch 11  loss=1.4780  acc=98.54%
Epoch 12  loss=1.4777  acc=98.55%
Epoch 13  loss=1.4773  acc=98.58%
Epoch 14  loss=1.4770  acc=98.61%
Epoch 15  loss=1.4768  acc=98.62%
Epoch 16  loss=1.4765  acc=98.65%
Epoch 17  loss=1.4763  acc=98.66%
